In [ ]:
import socket, torch
print("Máy:", socket.gethostname())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
!nvidia-smi -L


Máy: 3695663f2d8d
GPU: Tesla T4
GPU 0: Tesla T4 (UUID: GPU-71a7d123-3204-82b6-b4ff-59de9bec47ed)


# Practice 2 — Pre-trained neural network architectures (CIFAR-10)

7 bước của đề (mỗi bước một ô), sau đó 3 Exercises: so sánh model, chỉnh hyperparameters, TensorBoard.


## 1. Environment Setup

In [ ]:
!pip install torch torchvision tensorboard

import os
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## 2. Load a Pre-trained Model

`weights="DEFAULT"` tải trọng số pre-trained trên ImageNet (cách viết mới của `pretrained=True`).

In [ ]:
resnet18 = torchvision.models.resnet18(weights="DEFAULT")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 150MB/s]


## 3. Explore the Model Architecture

In [ ]:
print(resnet18)

for name, p in resnet18.named_parameters():
    print(name, tuple(p.shape))

print("Tổng số tham số:", sum(p.numel() for p in resnet18.parameters()))

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## 4. Adapt the Model for a Specific Task

Freeze các lớp phía trước (transfer learning) và thay lớp cuối để có 10 đầu ra cho CIFAR-10.
Hàm `get_model` làm việc này cho cả ResNet, DenseNet và VGG (dùng ở các Exercises).

In [ ]:
def get_model(name):
    model = getattr(torchvision.models, name)(weights="DEFAULT")
    for p in model.parameters():
        p.requires_grad = False                      # freeze
    # (muốn fine-tune: mở khoá vài lớp cuối, vd. p.requires_grad = True cho model.layer4)

    if name.startswith("densenet"):
        model.classifier = nn.Linear(model.classifier.in_features, 10)
    elif name.startswith("vgg"):
        model.classifier[6] = nn.Linear(4096, 10)
    else:                                            # resnet
        model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(device)


model = get_model("resnet18")
print(model.fc)

Linear(in_features=512, out_features=10, bias=True)


## 5. Prepare the Data

Server gốc của CIFAR-10 (Đại học Toronto) thường tải rất chậm trên Colab, nên ô đầu tiên tải trước đúng file
`cifar-10-python.tar.gz` (bản sao nguyên vẹn, torchvision vẫn kiểm tra MD5) từ Hugging Face. Ô này **không bắt buộc**:
bỏ qua thì ô sau vẫn tự tải từ server gốc, chỉ chậm hơn.

In [ ]:
os.makedirs("data", exist_ok=True)
!wget -O data/cifar-10-python.tar.gz https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/resolve/main/cifar-10-python.tar.gz

--2026-10-31 16:54:17--  https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/resolve/main/cifar-10-python.tar.gz
Resolving huggingface.co (huggingface.co)... 3.166.152.105, 3.166.152.110, 3.166.152.44, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.105|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6abae3560f675cbe4397cccc/72576deb7b0b1fa28fa6860119d69007df6cf6bbf244f871b824f31d1b3ddda5?response-content-type=application%2Fgzip&X-Xet-Cas-Uid=public&xip=daSjnre1CXY&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27cifar-10-python.tar.gz%3B+filename%3D%22cifar-10-python.tar.gz%22%3B&Expires=1793469257&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmFiYWUzNTYwZjY3NWNiZTQzOTdjY2NjLzcyNTc2ZGViN2IwYjFmYTI4ZmE2ODYwMTE5ZDY5MDA3ZGY2Y2Y2YmJmMjQ0Zjg3MWI4MjRmMzFkMWIzZGRkYTVcXD9yZXNwb25zZS1jb250ZW50LXR5cGU9KiZYLVhldC1DYXMtVWlkPXB1Ymxp

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)


def get_loaders(batch_size):
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)
    return trainloader, testloader


trainloader, testloader = get_loaders(32)
print("Train / test:", len(trainset), "/", len(testset))

Train / test: 50000 / 10000


## 6. Train the Model

Hàm `train(tên model, lr, batch size)` gồm: loss function + optimizer, vòng huấn luyện, và ghi loss vào **TensorBoard**.
Chạy mô hình gốc: **ResNet18, lr = 0.001, batch size = 32, 2 epoch**.

In [ ]:
def train(name, lr, batch_size, epochs=2):
    run_name = f"{name}_lr{lr}_bs{batch_size}"
    torch.manual_seed(42)
    model = get_model(name)
    trainloader, _ = get_loaders(batch_size)
    writer = SummaryWriter(f"runs/{run_name}")

    # Loss function and optimizer (chỉ cập nhật các tham số chưa bị freeze)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD([p for p in model.parameters() if p.requires_grad], lr=lr, momentum=0.9)

    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 100 == 99:                        # in loss mỗi 100 mini-batch
                avg_loss = running_loss / 100
                print(f"[{run_name}] epoch {epoch + 1}, batch {i + 1}: loss {avg_loss:.3f}")
                writer.add_scalar("Loss/train", avg_loss, epoch * len(trainloader) + i + 1)
                running_loss = 0.0

    writer.close()
    print("Finished Training")
    return model


resnet = train("resnet18", lr=0.001, batch_size=32)

[resnet18_lr0.001_bs32] epoch 1, batch 100: loss 2.216
[resnet18_lr0.001_bs32] epoch 1, batch 200: loss 1.876
[resnet18_lr0.001_bs32] epoch 1, batch 300: loss 1.810
[resnet18_lr0.001_bs32] epoch 1, batch 400: loss 1.772
[resnet18_lr0.001_bs32] epoch 1, batch 500: loss 1.745
[resnet18_lr0.001_bs32] epoch 1, batch 600: loss 1.734
[resnet18_lr0.001_bs32] epoch 1, batch 700: loss 1.693
[resnet18_lr0.001_bs32] epoch 1, batch 800: loss 1.690
[resnet18_lr0.001_bs32] epoch 1, batch 900: loss 1.708
[resnet18_lr0.001_bs32] epoch 1, batch 1000: loss 1.695
[resnet18_lr0.001_bs32] epoch 1, batch 1100: loss 1.656
[resnet18_lr0.001_bs32] epoch 1, batch 1200: loss 1.636
[resnet18_lr0.001_bs32] epoch 1, batch 1300: loss 1.632
[resnet18_lr0.001_bs32] epoch 1, batch 1400: loss 1.666
[resnet18_lr0.001_bs32] epoch 1, batch 1500: loss 1.670
[resnet18_lr0.001_bs32] epoch 2, batch 100: loss 1.640
[resnet18_lr0.001_bs32] epoch 2, batch 200: loss 1.613
[resnet18_lr0.001_bs32] epoch 2, batch 300: loss 1.645
[res

## 7. Evaluate the Model

Tính accuracy trên tập test (10.000 ảnh).

In [ ]:
def evaluate(model):
    _, testloader = get_loaders(100)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            predicted = model(images).argmax(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


accuracy = {}
accuracy["resnet18, lr=0.001, bs=32"] = evaluate(resnet)
print(f"Accuracy trên test: {accuracy['resnet18, lr=0.001, bs=32']:.2f}%")

Accuracy trên test: 43.93%


## Exercises

### Exercise 1 — Experiment with different pre-trained models

So sánh **ResNet18** (đã chạy ở trên) với **DenseNet121**, cùng lr = 0.001 và batch size = 32.
(Muốn thử VGG: đổi tên thành `"vgg11"`.)

In [ ]:
densenet = train("densenet121", lr=0.001, batch_size=32)
accuracy["densenet121, lr=0.001, bs=32"] = evaluate(densenet)

for k, v in accuracy.items():
    print(f"{k:<30} {v:.2f}%")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 163MB/s] 


[densenet121_lr0.001_bs32] epoch 1, batch 100: loss 2.145
[densenet121_lr0.001_bs32] epoch 1, batch 200: loss 1.792
[densenet121_lr0.001_bs32] epoch 1, batch 300: loss 1.692
[densenet121_lr0.001_bs32] epoch 1, batch 400: loss 1.615
[densenet121_lr0.001_bs32] epoch 1, batch 500: loss 1.586
[densenet121_lr0.001_bs32] epoch 1, batch 600: loss 1.570
[densenet121_lr0.001_bs32] epoch 1, batch 700: loss 1.546
[densenet121_lr0.001_bs32] epoch 1, batch 800: loss 1.551
[densenet121_lr0.001_bs32] epoch 1, batch 900: loss 1.468
[densenet121_lr0.001_bs32] epoch 1, batch 1000: loss 1.489
[densenet121_lr0.001_bs32] epoch 1, batch 1100: loss 1.499
[densenet121_lr0.001_bs32] epoch 1, batch 1200: loss 1.492
[densenet121_lr0.001_bs32] epoch 1, batch 1300: loss 1.489
[densenet121_lr0.001_bs32] epoch 1, batch 1400: loss 1.492
[densenet121_lr0.001_bs32] epoch 1, batch 1500: loss 1.493
[densenet121_lr0.001_bs32] epoch 2, batch 100: loss 1.462
[densenet121_lr0.001_bs32] epoch 2, batch 200: loss 1.447
[densene

### Exercise 2 — Adjust the hyperparameters

Mỗi hyperparameter so sánh 2 giá trị trên ResNet18:
- Learning rate: 0.001 (gốc) vs **0.01**
- Batch size: 32 (gốc) vs **128**

In [ ]:
accuracy["resnet18, lr=0.01, bs=32"] = evaluate(train("resnet18", lr=0.01, batch_size=32))
accuracy["resnet18, lr=0.001, bs=128"] = evaluate(train("resnet18", lr=0.001, batch_size=128))

for k, v in accuracy.items():
    print(f"{k:<30} {v:.2f}%")

[resnet18_lr0.01_bs32] epoch 1, batch 100: loss 2.544
[resnet18_lr0.01_bs32] epoch 1, batch 200: loss 2.539
[resnet18_lr0.01_bs32] epoch 1, batch 300: loss 2.549
[resnet18_lr0.01_bs32] epoch 1, batch 400: loss 2.603
[resnet18_lr0.01_bs32] epoch 1, batch 500: loss 2.751
[resnet18_lr0.01_bs32] epoch 1, batch 600: loss 2.673
[resnet18_lr0.01_bs32] epoch 1, batch 700: loss 2.678
[resnet18_lr0.01_bs32] epoch 1, batch 800: loss 2.616
[resnet18_lr0.01_bs32] epoch 1, batch 900: loss 2.789
[resnet18_lr0.01_bs32] epoch 1, batch 1000: loss 2.634
[resnet18_lr0.01_bs32] epoch 1, batch 1100: loss 2.756
[resnet18_lr0.01_bs32] epoch 1, batch 1200: loss 2.631
[resnet18_lr0.01_bs32] epoch 1, batch 1300: loss 2.656
[resnet18_lr0.01_bs32] epoch 1, batch 1400: loss 2.784
[resnet18_lr0.01_bs32] epoch 1, batch 1500: loss 2.653
[resnet18_lr0.01_bs32] epoch 2, batch 100: loss 2.681
[resnet18_lr0.01_bs32] epoch 2, batch 200: loss 2.683
[resnet18_lr0.01_bs32] epoch 2, batch 300: loss 2.796
[resnet18_lr0.01_bs32]

### Exercise 3 — Use TensorBoard to monitor the training process

Loss của mỗi lần chạy được ghi vào thư mục `runs/`. Mở TensorBoard để xem và so sánh các đường loss.
(Ở máy cá nhân có thể chạy `tensorboard --logdir runs` trong terminal rồi mở http://localhost:6006.)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

Output hidden; open in https://colab.research.google.com to view.